In [30]:
import os
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans

FILE_NAME = "p0.8_mu0.2_4"
emb_path = f"../pretrain/{FILE_NAME}.npy"
#emb_path = '/Users/acw721/Desktop/research/linkstream/pretrain/node_level_feat/node_feat.npy'
interaction_path = f"../syn_data/{FILE_NAME}.csv"
K = 5

X = np.load(emb_path).astype(np.float32)   # shape: [num_nodes, dim]


node_ids = np.arange(X.shape[0])

kmeans = KMeans(n_clusters=K, n_init=10, random_state=0)
labels = kmeans.fit_predict(X)

node2label = pd.DataFrame({
    "node": node_ids,
    "label": labels
})

inter_df = pd.read_csv(interaction_path)
inter_df = inter_df[["source", "destination", "timestamp"]].copy()

inter_df["source"] = inter_df["source"].astype(int)
inter_df["destination"] = inter_df["destination"].astype(int)

src_label_df = node2label.rename(columns={
    "node": "source",
    "label": "source_commu"
})
result_df = inter_df.merge(src_label_df, on="source", how="left")

dst_label_df = node2label.rename(columns={
    "node": "destination",
    "label": "destination_commu"
})
result_df = result_df.merge(dst_label_df, on="destination", how="left")

result_df = result_df[
    ["source", "destination", "timestamp", "source_commu", "destination_commu"]
].copy()

out_dir = os.path.join("../results", FILE_NAME)
os.makedirs(out_dir, exist_ok=True)

out_path = os.path.join(out_dir, "ctdne.csv")
result_df.to_csv(out_path, index=False)

print(result_df.head(20))
print(f"saved to {out_path}")
print("rows =", len(result_df))
print("missing source_commu =", result_df["source_commu"].isna().sum())
print("missing destination_commu =", result_df["destination_commu"].isna().sum())

    source  destination  timestamp  source_commu  destination_commu
0        0           49          5             4                  4
1        6           32         11             4                  0
2       21           32         18             1                  0
3       12           25         25             1                  1
4       40           45         31             3                  2
5        3           15         37             1                  4
6       10           27         43             2                  2
7        5           28         52             2                  2
8       23           40         60             4                  3
9        5           32         63             2                  0
10       0            7         70             4                  0
11       4           18         77             2                  2
12      23           48         83             4                  4
13       5           22         87             2

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans

SYN_DATA_DIR = "../syn_data"
PRETRAIN_DIR = "../pretrain"
RESULTS_DIR = "results"
K = 5

csv_files = sorted(glob.glob(os.path.join(SYN_DATA_DIR, "*.csv")))
print(f"Found {len(csv_files)} csv files.")

for interaction_path in csv_files:
    FILE_NAME = os.path.splitext(os.path.basename(interaction_path))[0]
    emb_path = os.path.join(PRETRAIN_DIR, f"{FILE_NAME}.npy")

    print(f"\nProcessing: {FILE_NAME}")

    if not os.path.exists(emb_path):
        print(f"  Skip: embedding file not found -> {emb_path}")
        continue

    X = np.load(emb_path).astype(np.float32)   # shape: [num_nodes, dim]
    node_ids = np.arange(X.shape[0])

    kmeans = KMeans(n_clusters=K, n_init=10, random_state=0)
    labels = kmeans.fit_predict(X)

    node2label = pd.DataFrame({
        "node": node_ids,
        "label": labels
    })

    inter_df = pd.read_csv(interaction_path)
    inter_df = inter_df[["source", "destination", "timestamp"]].copy()
    inter_df["source"] = inter_df["source"].astype(int)
    inter_df["destination"] = inter_df["destination"].astype(int)

    src_label_df = node2label.rename(columns={
        "node": "source",
        "label": "source_commu"
    })
    result_df = inter_df.merge(src_label_df, on="source", how="left")

    dst_label_df = node2label.rename(columns={
        "node": "destination",
        "label": "destination_commu"
    })
    result_df = result_df.merge(dst_label_df, on="destination", how="left")

    result_df = result_df[
        ["source", "destination", "timestamp", "source_commu", "destination_commu"]
    ].copy()

    out_dir = os.path.join(RESULTS_DIR, FILE_NAME)
    os.makedirs(out_dir, exist_ok=True)

    out_path = os.path.join(out_dir, "ctdne.csv")
    result_df.to_csv(out_path, index=False)

    print(result_df.head(5))
    print(f"  saved to {out_path}")
    print(f"  rows = {len(result_df)}")
    print(f"  missing source_commu = {result_df['source_commu'].isna().sum()}")
    print(f"  missing destination_commu = {result_df['destination_commu'].isna().sum()}")

print("\nDone.")

Found 100 csv files.

Processing: p0.85_mu0.05_1
   source  destination  timestamp  source_commu  destination_commu
0      15           20          9             0                  0
1       4           25         15             2                  2
2      24           32         21             0                  0
3       8           10         28             4                  4
4      16           49         35             2                  1
  saved to results/p0.85_mu0.05_1/ctdne.csv
  rows = 473
  missing source_commu = 0
  missing destination_commu = 0

Processing: p0.85_mu0.05_2
   source  destination  timestamp  source_commu  destination_commu
0       5           11          6             2                  2
1       4           11         13             2                  2
2      17           35         20             1                  1
3       0           32         26             3                  4
4      17           28         32             1                  1
  s